# BeautifulSoup (BS4)
- [Documentación](https://beautiful-soup-4.readthedocs.io/en/latest/)
  
Es la herramienta estándar para navegar y extraer datos de documentos HTML y XML. No es un navegador (como Selenium), sino un parser (analizador).

## Instalación
``` bash
pip install beautifulsoup4 lxml requests
``` 

- **beautifulsoup4**: Librería principal (bs4) que facilita la navegación y búsqueda dentro de la estructura HTML o XML, permitiendo extraer texto, enlaces y tablas.
  
- **lxml**: Es un motor (parser) de alto rendimiento que BeautifulSoup utiliza para interpretar el HTML de manera rápida y eficiente.

- **requests**: Librería esencial para enviar solicitudes HTTP (como GET o POST) y obtener el contenido HTML de una página web para su posterior análisis. 

### Estructura básica
``` Python
import requests
from bs4 import BeautifulSoup

# URL objetivo que queremos consultar
url = "https://example.com"

# Headers HTTP que enviamos junto a la solicitud
headers = {"User-Agent": "Mozilla/5.0"} 
# El User-Agent simula un navegador real
# 💡 Tip: muchos sitios bloquean requests sin User-Agent
# para evitar bots o scraping automatizado

response = requests.get(url, headers=headers)

# Creamos el objeto 'soup' a partir del HTML recibido
soup = BeautifulSoup(response.text, "lxml")
# BeautifulSoup transforma el HTML en una estructura navegable
# "lxml" es el parser:
# - Más rápido que html.parser
# - Más robusto ante HTML mal formado
```

## 🔍 Métodos de Búsqueda 

| Método | Retorno | Uso |
|--------------|--------------|--------------|
| `find()` | Un solo objeto | Encuentra la primera coincidencia |
| `find_all()` | Una lista | Encuentra todas las coincidencias |
| `select_one()` | Un solo objeto | Usa selectores CSS (estilo Senior) |
| `select()` | Una lista | Usa selectores CSS para múltiples elementos |

### Ejemplos de Búsqueda

```Python
# Por etiqueta
titulo = soup.find("h1")

# Por clase (usa class_ porque class es palabra reservada en Python)
precios = soup.find_all("span", class_="price-tag")

# Por ID
footer = soup.find(id="main-footer")

# Búsqueda anidada
contenedor = soup.find("div", class_="container")
links = contenedor.find_all("a")
```

### Selectores CSS - `.select()`
  
**`.select()`** permite queries más complejas y es más legible.
  
```Python
# Encontrar todos los links dentro de una lista con clase 'menu'
links = soup.select("ul.menu > li > a")

# Encontrar un elemento con varios atributos
item = soup.select_one("div.product[data-id='123']")
```

### Extracción de Datos
Una vez que tienes el elemento, necesitas sacar la información:

```Python
element = soup.find("a", class_="link-external")

# 1. El texto visible (.strip() para limpiar espacios en blanco)
texto = element.get_text(strip=True) 

# 2. Atributos (se manejan como diccionarios)
url_destino = element.get("href")
data_id = element["data-info"] # Ojo: lanza KeyError si no existe. Mejor usar .get()
```

### Navegación por el Árbol (DOM Traversal)
  
A veces el dato no tiene ID ni Clase, y debes moverte respecto a otros elementos.
  
- .parent: Sube un nivel.
  
- .find_next_sibling(): Busca el siguiente elemento al mismo nivel (muy útil en tablas).
  
- .children: Itera sobre los hijos directos.

#### Ejemplo Senior: Extracción de una Tabla de Precios
  
Este ejemplo incluye manejo de errores y limpieza, esenciales para un pipeline de datos.

  
```Python
def extract_table_data(soup: BeautifulSoup) -> list[dict]:
    results = []
    table = soup.select_one("table#market-data")
    
    if not table:
        return []

    # Saltamos el header (th) y vamos por las filas (tr)
    for row in table.find_all("tr")[1:]:
        cols = row.find_all("td")
        if len(cols) >= 2:
            data = {
                "producto": cols[0].get_text(strip=True),
                "precio": float(cols[1].get_text(strip=True).replace("$", "").replace(",", "")),
                "stock": "disponible" in cols[2].get("class", [])
            }
            results.append(data)
    
    return results
```

---

## 💡 Tips de Senior (Best Practices)
1. Evita el "`AttributeError`": Siempre verifica si tu búsqueda devolvió algo antes de pedir el texto.
  
    - **Mal:** `soup.find("h1").text` (si no hay h1, el script muere).

    - **Bien:** `h1 = soup.find("h1")`; `text = h1.get_text() if h1 else "N/A"`.
  
2. Limpia siempre el texto: El HTML suele venir con saltos de línea \n y espacios extra. Usa get_text(strip=True).
  
3. Respeta el robots.txt: Antes de scrapear, revisa example.com/robots.txt. Como ingeniero, la ética y la legalidad son parte de tu responsabilidad técnica.
  
4. No satures el servidor: Si vas a scrapear miles de páginas, usa time.sleep(1) entre peticiones. Si el servidor te bloquea la IP, tu pipeline dejará de funcionar.
  
5. Data Quality con Pydantic: Una vez extraído el dato con BeautifulSoup, pásalo por un modelo de Pydantic. Si el precio no es un número, descarta ese registro antes de que llegue a tu base de datos SQL.